:::{admonition} 5.1: バッチ推定
:class: note

## 5.1.1 目的  
本課題では **バッチ最小二乗法（Batch LS）** を用いて低軌道衛星の軌道を推定し，  力学モデルの精度と観測幾何（地上局配置）が推定結果に与える影響を評価する。   

1. **非線形力学モデル（J2 + 空気抵抗）** を定式化し、与えられた状態変数と対応する**状態遷移行列（STM）** を同時に数値積分する。  
2. **双方向レンジ／レンジレート観測**をモデル化し、状態変数によるヤコビ行列（$H$行列）を導出・実装する。  
3. 正規方程式を組み立てて、初期状態および物理パラメータの **バッチ推定（Batch LS）** ができる。  
4. 観測量と推定値の**残差 RMS・共分散** を用いてモデル／幾何の妥当性を判断し、適宜モデルの改善について考察を実施する。

In [ ]:
# ----------------------------------------------------------------------
# 0. Imports
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D      # noqa: F401
from scipy.integrate import solve_ivp        # ode45 eq.  :contentReference[oaicite:8]{index=8}
from scipy.stats import norm                 # PDF         :contentReference[oaicite:9]{index=9}
from scipy.linalg import block_diag          # block diag  :contentReference[oaicite:10]{index=10}
import sympy as sp                           # lambdify    :contentReference[oaicite:11]{index=11}

## 5.1.2 前提条件
### 5.1.2.1 観測量
観測量は、以下のようにtxtファイル形式で与えられる。
| ファイル          | 内容                                    |
|------------------|-----------------------------------------|
| `project.txt`    | 観測データ：<br>時刻 \(t_k\) [s]，地上局 ID (101/337/394)，レンジ [m]，レンジレート [m/s] |
> **注意**：観測レンジ／レンジレートは *メートル系* で与えられている。  

In [ ]:
# ----------------------------------------------------------------------
# 1. Parse Data  (MATLAB: readmatrix)
# ----------------------------------------------------------------------
fname = Path("/Users/Tetsuya/Desktop/StatOD/data/project.txt")
df = pd.read_csv(fname, delim_whitespace=True, header=None)   # :contentReference[oaicite:12]{index=12}
time          = df.iloc[:, 0].to_numpy(dtype=float)
stationID     = df.iloc[:, 1].to_numpy(dtype=int)
rng_obs       = df.iloc[:, 2].to_numpy(dtype=float) / 1e3
rng_rate_obs  = df.iloc[:, 3].to_numpy(dtype=float) / 1e3

trackingData = dict(time=time,
                    stationID=stationID,
                    range=rng_obs,
                    rangeRate=rng_rate_obs)

### 5.1.2.2 初期状態量 / ECI(Earth Centric Inertial)座標系

| 成分 | 記号 | 値 | 単位 |
|------|------|------|------|
| 位置 $x$ | $x_0$ | $757.7000$ | $[km]$ |
| 位置 $y$ | $y_0$ | $5222.6070$ | $[km]$ |
| 位置 $z$ | $z_0$ | $4851.5000$ | $[km]$ |
| 速度 $vx$ | $\dot x_0$ | $2.21321$ | $[km s^{-1}]$ |
| 速度 $vy$ | $\dot y_0$ | $4.67834$ | $[km s^{-1}]$ |
| 速度 $vz$ | $\dot z_0$ | $−5.37130$ | $[km s^{-1}]$ |

In [ ]:
x0 = np.array([757.7, 5222.607, 4851.5, 2.21321, 4.67834, -5.3713])

### 1.3 物理パラメータ初期値

| パラメータ | 記号 | 初期値 | 単位 | 事前標準偏差（推奨） |
|------------|------|-------:|------|----------------------|
| 地球重力定数 | $\mu$ | $398 600.4415$ | $[km^3 s^{-2}]$ | $1.0 \times 10^{5}$ |
| J1 項 | $J_1$ | $0$ | — | 固定 |
| J2 項 | $J_2$ | $1.082626925638815 × 10^{-3}$ | — | $1.0\times 10^{-7}$ |
| ドラッグ係数 | $C_D$ | $2.0$ | — | $0.1$ |
| 断面積 | $A$ | $3.0$ (→ $3 \times 10^{-6} km^2 $)| $[m^2]$ | 固定 |
| 質量 | $m$ | $970$ | $[kg]$ | 固定 |
| 基準密度 | $\rho_0$ | $3.614 \times 10^{-4}$ | $[kg m^{-3}]$ | 固定 |
| 基準高度 | $h_0$ | $700$ | $[km]$ | 固定 |
| スケールハイト | $H$ | $88.667$ | $[km]$ | 固定 |

In [ ]:
# ----------------------------------------------------------------------
# 2. Constants & Initial State
# ----------------------------------------------------------------------
coeffs = np.array([0.0, 1.082626925638815e-3])
R_EARTH = 6378.1363          # km
MU      = 398600.4415        # km^3/s^2
L_MAX   = 2
AREA    = 3e-6
MASS    = 970.0
CD      = 2.0
PARAM_MEAS = 9  

### 1.3 地上局座標 / ECEF(Earth Centric Earth Fixed)座標系

| GS ID | $x_{GS} [km]$ | $y_{GS} [km]$ | $z_{GS} [km]$ |
|-------|---------------:|---------------:|---------------:|
| $101$ | $−5127.5100$ | $−3794.1600$ | $0.0000$ |
| $337$ | $3860.9100$ | $3238.4900$ | $3898.0940$ |
| $394$ | $549.5050$ | $−1380.8720$ | $6182.1970$ |

In [ ]:
STATIONS_ECEF0 = {
    101: np.array([-5127.5100, -3794.1600, 0.0]),
    337: np.array([ 3860.9100,  3238.4900, 3898.0940]),
    394: np.array([  549.5050, -1380.8720, 6182.1970]),
}

### 2. 状態ベクトル
衛星の位置・速度、各種物理パラメータ、3つの地上局位置を表す、以下のような状態変数を考える。
$$
\mathbf x
=\bigl[\,\underbrace{\mathbf r,\mathbf v}_{6},\;
        μ,\;J_1, J_2,\;C_D,\;
        \mathbf r_{GS1},\;\mathbf r_{GS2},\;\mathbf r_{GS3}\bigr]^{\mathsf T}
\in\mathbb R^{19}.
$$

$STM$ は $19\times 19$ とし、初期値は単位行列である。
これら $19 + 19\times 19 = 380$ 次元の状態ベクトルを考え、数値積分する。

In [ ]:
state0 = np.hstack([x0, MU, coeffs, CD,
                    STATIONS_ECEF0[101],
                    STATIONS_ECEF0[337],
                    STATIONS_ECEF0[394],
                    np.eye(19).ravel()])   # 6+1+2+1+3*3 = 19 states

### 3. 観測モデル
観測量は、3種類の地上局で得られるレンジ、レンジレートとする。ただし、伝播遅延、相対論・大気遅延・重力赤方偏移の影響などは無視することとする。

また、状態変数によるヤコビアンを解析的に導出し，**2 × 19** の行列 $\mathbf H_k$ を実装すること。

---

### 4. 推定ケース

| Case | 推定する地上局座標 | 初期分散設定　$\sigma^2\$ |
|------|-------------------|-----------------------------|
| **GS1** | GS2, GS3        | 推定局: $1 \times10^{+0}\,\text{km}^2$ <br>固定局(GS1): $1 \times10^{-6}\,\text{km}^2\$ |
| **GS2** | GS1, GS3        | 同上（GS2 固定） |
| **GS3** | GS1, GS2        | 同上（GS3 固定） |
| **None**| 3 局すべて       | 3 局とも $1 \times10^{+0}\,\text{km}^2$ |

> 非推定パラメータ（μ, J2, CD 等）の初期分散は `constants.txt` 推奨値を用いよ。  
> 観測雑音共分散は $\sigma_\rho=1\times10^{-2}\,\text{m}$、$\sigma_{\dot\rho}=1\times10^{-3}\,\text{m/s}$ とする。

---

### 5. 検討事項
 
1. 力学／観測モデルの数式  
2. STM を含む ODE 実装の説明  
3. バッチ正規方程式の組立て手順  
4. 各ケースの結果  
   * 初期状態量を伝播した3次元軌道図（ECI）  
   * pre／post-fit 残差プロット（ρ, \(\dotρ\)）  
   * RMS 値表  
   * 推定パラメータと 1‑σ 不確定性  
5. 考察  
   * どの局を固定すべきか？理由は？  
   * モデル誤差（高次重力，SRP 等）が残差に与える影響の推察  
   * 反復バッチ推定の必要性有無

### $J_2$ 摂動加速度の導出

#### 1.  理想球 vs. 扁平地球

- **理想球（質量 $M$ が中心に集中）**  
地球が完全な球であれば、重力ポテンシャルは、
$$
  U_0(r)= -\frac{GM}{r},\qquad r=\sqrt{x^{2}+y^{2}+z^{2}}.
$$

- **実際の地球**  
しかし、実際の地球は*赤道半径* $a=6378\text{ km}$、*極半径* $b=6357\text{ km}$ から分かるように、[扁平楕円体](https://ja.wikipedia.org/wiki/%E6%89%81%E7%90%83)である。
つまり、赤道方向にわずかにふくらみ，極方向にへこんでいる。この扁平度は無次元量$f$で表される。  
$$
  f=\frac{a-b}{a}\approx\frac1{298}.
$$

##### 2.  重力ポテンシャルの球面調和展開

地球の質量分布を球面調和級数（ゾーナル項のみ）で表すと  

$$
U(r,\phi)= -\frac{GM}{r}
\Biggl[
  1-\sum_{\ell=1}^{\infty}
    \Bigl(\frac{R_\oplus}{r}\Bigr)^{\!\ell}
    \;J_\ell\;P_\ell(\sin\phi)
\Biggr],
$$

- $\phi$：測地緯度（ECI 基準なら $\sin\phi = z/r$）  
- $P_\ell$：Legendre 多項式  
- $J_\ell$：**ゾーナル係数**  
  - $J_1$：質量中心のズレ（一般的に0とすることが多い）
  - $J_2$：扁平性を示す。

$J_2$が突出して大きく，他の$J_l(l>=3)$ は数桁小さい。そのため低高度では$J_2$項のみで95%以上の非球対項を説明できる。


実測値は、
$$
J_2 \;=\; 1.082\,626\,925... \times 10^{-3}.
$$


#### 3.  $J_2$ 項のみ抽出

$$
\begin{aligned}
U_{J2}
&= -\frac{GM}{r}
    \Bigl(\frac{R_\oplus}{r}\Bigr)^{\!2}
    J_2\,P_2(\sin\phi) ,\\[6pt]
P_2(\sin\phi)
&=\tfrac12\bigl(3\sin^{2}\phi-1\bigr)
  =\tfrac12\!\Bigl(1-\frac{3z^{2}}{r^{2}}\Bigr).
\end{aligned}
$$

整理すると  

$$
U_{J2}= -\frac{GMJ_2R_\oplus^{2}}{2r^{3}}
        \Bigl(1-\frac{3z^{2}}{r^{2}}\Bigr).
$$


#### 4.  勾配を取って加速度へ

重力加速度は $\mathbf a = -\nabla U$ で計算される。したがって、

$$
\boxed{
\mathbf a_{J2}=
-\frac{3GMJ_2R_\oplus^{2}}{2r^{5}}
\begin{bmatrix}
x\left(1-\dfrac{5z^{2}}{r^{2}}\right) \\[6pt]
y\left(1-\dfrac{5z^{2}}{r^{2}}\right) \\[6pt]
z\left(3-\dfrac{5z^{2}}{r^{2}}\right)
\end{bmatrix}}
$$

- $r^{-5}$ で減衰：二体重力の $r^{-2}$ に比べ $r^{-3}$ だけ速く弱まる  
- $x,y$ 成分：$\bigl(1-\tfrac{5z^{2}}{r^{2}}\bigr)$ が非球対性を与える  
- $z$ 成分：$\bigl(3-\tfrac{5z^{2}}{r^{2}}\bigr)$ が極方向へ戻す成分

#### 5.  直感的な影響

1. **軌道面の歳差（Ω̇）**  
   赤道膨張により軌道面がゆっくり回転。  
2. **近地点引数の回転（ω̇）**  
   赤道傾斜によるトルクで近地点が移動。  
3. **低高度ほど顕著**  
   $J_2$ は $r^{-5}$ 依存なので LEO で支配的、  
   GEO ではほぼ無視できる。

#### 6.  まとめ

1. 地球の“赤道膨張”という 2 次の形状ゆがみを  
   **ゾーナル係数 $J_2$** でパラメータ化。  
2. ポテンシャル $U = U_0 + U_{J2}$ として追加。  
3. 勾配 $-\nabla U_{J2}$ を取ることで  
   上式の **$J_2$ 摂動加速度** に到達。  
4. これを二体重力に加えるだけで LEO～MEO の  
   非球対重力効果の大部分が捕捉できる。


In [ ]:
def precompute_zonal_sph(R_val: float, l_max: int):
    x, y, z, vx, vy, vz, mu = sp.symbols('x y z vx vy vz mu')
    coeffs_sym = sp.symbols(f'J0:{l_max}')
    r = sp.sqrt(x**2 + y**2 + z**2)
    phi = sp.asin(z/r)
    U = mu/r
    for l in range(1, l_max+1):
        U -= mu/r * coeffs_sym[l-1] * (R_val/r)**l * sp.legendre(l, sp.sin(phi))
    acc = sp.Matrix([sp.diff(U, v) for v in (x, y, z)])
    aug = sp.Matrix.vstack(acc,
                           sp.zeros(1,1),          # dµ placeholder
                           sp.zeros(l_max, 1))     # dJₗ placeholder
    acc_f = sp.lambdify((x, y, z, mu, coeffs_sym), aug, 'numpy')  # :contentReference[oaicite:13]{index=13}
    f_full = sp.Matrix.vstack(sp.Matrix([vx,vy,vz]), aug)
    A = f_full.jacobian([x,y,z,vx,vy,vz,mu,*coeffs_sym])
    A_f = sp.lambdify((x,y,z,vx,vy,vz,mu,coeffs_sym), A, 'numpy')
    return acc_f, A_f

### 観測量と観測モデルの定式化

本課題で扱う観測は **レンジ** と **レンジレート** の2種類のみです。  
いずれも「**地上局 ↔ 衛星**」間の瞬時幾何距離・相対速度に基づき、電波往復時間や往復位相から計算された値と考えます（大気・相対論補正などの細かい修正項は今回は無視）。  
以下では **ECI 座標系** を基準に、数式、偏微分（$H$行列）の導出手順を詳述します。

---

#### 1. 座標系と記号

| 記号 | 定義 (ECI 基準) |
|------|-----------------|
| $\mathbf r$ | 衛星位置ベクトル $[x,\,y,\,z]^{\mathsf T}$ |
| $\mathbf v$ | 衛星速度ベクトル $[\dot x,\,\dot y,\,\dot z]^{\mathsf T}$ |
| $\mathbf r_s(t)$ | 地上局位置ベクトル |
| $\mathbf v_s(t)$ | 地上局速度ベクトル |
| $\boldsymbol ω_\oplus$ | 地球自転角速度 $[0,\,0,\,ω_\oplus]^{\mathsf T}$ |

地上局の ECI 座標は **ECEF → ECI** 回転で

$$
\mathbf r_s(t)=
\mathbf C_{\text{ECI}\leftarrow\text{ECEF}}\!\bigl(θ(t)\bigr)\,\mathbf r_{s,0},
\qquad
θ(t)=ω_\oplus t,
$$

速度は

$$
\mathbf v_s(t)=\boldsymbol ω_\oplus \times \mathbf r_s(t).
$$

---

#### 2. 観測モデル

##### 2.1 レンジ $\rho$

$$
\rho =\bigl\|\mathbf r-\mathbf r_s\bigr\| 
      =\sqrt{(\mathbf r-\mathbf r_s)\!\cdot(\mathbf r-\mathbf r_s)}.
$$

##### 2.2 レンジレート $\dot\rho$

$$
\dot\rho
      =\frac{(\mathbf r-\mathbf r_s)\cdot(\mathbf v-\mathbf v_s)}{\rho}.
$$

ただし、以下の仮定を置いている。
> - 往復遅延（lc / 2）の効果は「即時距離」として近似  
> - 相対論・大気遅延・重力赤方偏移などは無視  

---

#### 3. 線形化と設計行列（偏微分）

観測ベクトル  
$$
\mathbf h(\mathbf x,t)=
\begin{bmatrix}
\rho \\ \dot\rho
\end{bmatrix},\qquad
$$

$$
\mathbf x
=\bigl[\,\underbrace{\mathbf r,\mathbf v}_{6},\;
        μ,\;J_1, J_2,\;C_D,\;
        \mathbf r_{GS1},\;\mathbf r_{GS2},\;\mathbf r_{GS3}\bigr]^{\mathsf T}
\in\mathbb R^{19}.
$$

##### 3.1 位置・速度に関する偏微分

まず $\mathbf Δ=\mathbf r-\mathbf r_s,\quad \mathbf δ=\mathbf v-\mathbf v_s$ とおくとレンジに関しては、

$$
\frac{\partial\rho}{\partial\mathbf r}
      =\frac{\mathbf Δ^{\mathsf T}}{\rho},\quad
\frac{\partial\rho}{\partial\mathbf v}
      =\mathbf 0_{1\times3}.
$$

続いて、レンジレートに関しては、

$$
\dot\rho =\frac{\mathbf Δ\cdot\mathbf δ}{\rho}
       \quad\Longrightarrow\quad
\begin{aligned}
\frac{\partial\dot\rho}{\partial\mathbf r}
 &=
\frac{1}{\rho}
\Bigl(\mathbf δ^{\mathsf T}-\dot\rho\,\frac{\mathbf Δ^{\mathsf T}}{\rho}\Bigr),\\[6pt]
\frac{\partial\dot\rho}{\partial\mathbf v}
 &=\frac{\mathbf Δ^{\mathsf T}}{\rho}.
\end{aligned}
$$

##### 3.2 地上局座標に関する偏微分

ECEF → ECI 回転を $\mathbf C(θ)$ とおくと  
$$
\mathbf r_s = \mathbf C\,\mathbf r_{s,0}
$$  
ECI での微小変位は
$$
\mathrm d\mathbf r_s = \mathbf C\,\mathrm d\mathbf r_{s,0}
$$

$$
\frac{\partial\rho}{\partial\mathbf r_{s,0}}
     = -\frac{\partial\rho}{\partial\mathbf r}\,\mathbf C,
\qquad
\frac{\partial\dot\rho}{\partial\mathbf r_{s,0}}
     = -\frac{\partial\dot\rho}{\partial\mathbf r}\,\mathbf C.
$$

> 実装では $\mathbf C$ が**直交行列**なので転置 = 逆行列が用いられる  
> $\mathbf C^{-1}=\mathbf C^{\mathsf T}$。

---

##### 3.3  $\mathbf H_k$

2 行 19 列（本課題の状態次元）でブロック配置すると

$$
\mathbf H_k=
\begin{bmatrix}
\displaystyle\frac{\partial\rho}{\partial\mathbf r} &
\mathbf 0 &
\mathbf 0 &
-\displaystyle\frac{\partial\rho}{\partial\mathbf r}\,\mathbf C & \cdots
\\[10pt]
\displaystyle\frac{\partial\dot\rho}{\partial\mathbf r} &
\displaystyle\frac{\partial\dot\rho}{\partial\mathbf v} &
\mathbf 0 &
-\displaystyle\frac{\partial\dot\rho}{\partial\mathbf r}\,\mathbf C & \cdots
\end{bmatrix}.
$$

* $\mathbf 0$ は μ, J2, C_D など動的パラメータに対する直接偏微分（本課題ではゼロ）。  
* $\cdots$ は推定対象でない他局座標列に対応する 0 ブロック。

---

#### 4. 観測ノイズと共分散
観測雑音は平均 0，分散既知のガウス白色雑音と仮定する。
今回は  

$$
  \sigma_\rho = 10^{-2}\,\text{m},\qquad
  \sigma_{\dot\rho} = 10^{-3}\,\text{m s}^{-1}.
$$

として、共分散行列は以下のように定義する。

$$
    \mathbf R = \operatorname{diag}\!\bigl(\sigma_\rho^{2},\;\sigma_{\dot\rho}^{2}\bigr).
$$

#### 5. 実装メモ

1. **行列形で計算**することで `NumPy` のブロードキャストが有効になり、高速。  
2. 偏微分は常に **ECI 系** で評価し、地上局座標偏微分のみ $\mathbf C^{\mathsf T}$ で換算。  

---

#### 6. まとめ

* 観測モデルはユークリッド距離とその時間微分のみで単純だが、  **地球自転で動く地上局の位置・速度** を正しく扱う点が今回のポイント。  
* 線形化により得られる $\mathbf H_k$ はバッチ・カルマン双方で共通に使用できる。  
